## Imports

In [1]:
import numpy as np
import pandas as pd
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Data

In [2]:
# Create sample data
data = {
    "text": [
        "I feel very happy today",
        "Life is beautiful",
        "I am extremely sad",
        "I hate everything",
        "This is amazing",
        "I feel depressed",
        "I am excited about life",
        "I feel lonely and broken",
        "Today is wonderful",
        "I am stressed and anxious",
        "I love my friends",
        "Nobody understands me",
        "I feel hopeless",
        "Everything is going great",
        "I want to cry",
        "I am feeling fantastic"
    ],

    "sentiment": [
        "positive",
        "positive",
        "negative",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "negative",
        "positive",
        "negative",
        "positive"
    ]
}

df = pd.DataFrame(data)

print(df.head())

                      text sentiment
0  I feel very happy today  positive
1        Life is beautiful  positive
2       I am extremely sad  negative
3        I hate everything  negative
4          This is amazing  positive


## Preprocessing

In [3]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z\s]', '', text)
  return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']]

,text,clean_text
0,I feel very happy today,i feel very happy today
1,Life is beautiful,life is beautiful
2,I am extremely sad,i am extremely sad
3,I hate everything,i hate everything
4,This is amazing,this is amazing
5,I feel depressed,i feel depressed
6,I am excited about life,i am excited about life
7,I feel lonely and broken,i feel lonely and broken
8,Today is wonderful,today is wonderful
9,I am stressed and anxious,i am stressed and anxious


In [4]:
# Label Encoding
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['sentiment'])
df[['sentiment', 'label']]

,sentiment,label
0,positive,1
1,positive,1
2,negative,0
3,negative,0
4,positive,1
5,negative,0
6,positive,1
7,negative,0
8,positive,1
9,negative,0


In [5]:
# Tokenization
tokenizer = Tokenizer(
  num_words=1000,
  oov_token='<OOV>'
)

tokenizer.fit_on_texts(df['clean_text'])
word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(df['clean_text'])
sequences

[[2, 3, 10, 11, 6],
 [7, 4, 12],
 [2, 5, 13, 14],
 [2, 15, 8],
 [16, 4, 17],
 [2, 3, 18],
 [2, 5, 19, 20, 7],
 [2, 3, 21, 9, 22],
 [6, 4, 23],
 [2, 5, 24, 9, 25],
 [2, 26, 27, 28],
 [29, 30, 31],
 [2, 3, 32],
 [8, 4, 33, 34],
 [2, 35, 36, 37],
 [2, 5, 38, 39]]

In [6]:
# Padding
MAX_LEN = 10

X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = df['label']
X

array([[ 2,  3, 10, 11,  6,  0,  0,  0,  0,  0],
       [ 7,  4, 12,  0,  0,  0,  0,  0,  0,  0],
       [ 2,  5, 13, 14,  0,  0,  0,  0,  0,  0],
       [ 2, 15,  8,  0,  0,  0,  0,  0,  0,  0],
       [16,  4, 17,  0,  0,  0,  0,  0,  0,  0],
       [ 2,  3, 18,  0,  0,  0,  0,  0,  0,  0],
       [ 2,  5, 19, 20,  7,  0,  0,  0,  0,  0],
       [ 2,  3, 21,  9, 22,  0,  0,  0,  0,  0],
       [ 6,  4, 23,  0,  0,  0,  0,  0,  0,  0],
       [ 2,  5, 24,  9, 25,  0,  0,  0,  0,  0],
       [ 2, 26, 27, 28,  0,  0,  0,  0,  0,  0],
       [29, 30, 31,  0,  0,  0,  0,  0,  0,  0],
       [ 2,  3, 32,  0,  0,  0,  0,  0,  0,  0],
       [ 8,  4, 33, 34,  0,  0,  0,  0,  0,  0],
       [ 2, 35, 36, 37,  0,  0,  0,  0,  0,  0],
       [ 2,  5, 38, 39,  0,  0,  0,  0,  0,  0]], dtype=int32)

## Train Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Build Model

In [10]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=1000,
        output_dim=64,
        input_length=MAX_LEN
    )
)

# LSTM Layer
model.add(
    LSTM(64)
)

# Output Layer
model.add(
    Dense(1, activation='sigmoid')
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Train Model
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=4,
    validation_data=(X_test, y_test)
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 213ms/step - accuracy: 0.3333 - loss: 0.6971 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5000 - loss: 0.6931 - val_accuracy: 0.5000 - val_loss: 0.6928
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.5000 - loss: 0.6935 - val_accuracy: 0.5000 - val_loss: 0.6925
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5000 - loss: 0.6918 - val_accuracy: 0.5000 - val_loss: 0.6920
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.5000 - loss: 0.6909 - val_accuracy: 0.5000 - val_loss: 0.6913
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5833 - loss: 0.6887 - val_accuracy: 0.5000 - val_loss: 0.6900
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9167 - loss: 0.6858 - val_accuracy: 1.0000 - val_loss: 0.6880
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 1.0000 - loss: 0.6818 - val_accuracy: 1.0000 - val_loss: 0.6847

## Evaluation

In [12]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step - accuracy: 0.7500 - loss: 1.1694
Accuracy: 0.75


In [13]:
def predict_sentiment(text):

    # clean
    text = clean_text(text)

    # tokenize
    seq = tokenizer.texts_to_sequences([text])

    # pad
    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding='post'
    )

    # prediction
    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        sentiment = "POSITIVE"
    else:
        sentiment = "NEGATIVE"

    print(f"\nText: {text}")
    print(f"Prediction Score: {prediction:.4f}")
    print(f"Sentiment: {sentiment}")

In [14]:
predict_sentiment("I feel happy today")
predict_sentiment("I am very depressed")
predict_sentiment("Nobody loves me")
predict_sentiment("Life is wonderful")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step

Text: i feel happy today
Prediction Score: 0.0126
Sentiment: NEGATIVE
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step

Text: i am very depressed
Prediction Score: 0.9088
Sentiment: POSITIVE
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step

Text: nobody loves me
Prediction Score: 0.0072
Sentiment: NEGATIVE
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step

Text: life is wonderful
Prediction Score: 0.9955
Sentiment: POSITIVE
